In [ ]:
import pandas as pd
import numpy as np

# New Dataset
data = {
    'Study_Time': ['High','High','Medium','Low','Low','Low','Medium','High','High','Low','High','Medium','Medium','Low'],
    'Attendance': ['Good','Poor','Good','Good','Good','Poor','Poor','Good','Good','Good','Poor','Poor','Good','Poor'],
    'Assignments': ['Complete','Complete','Incomplete','Complete','Complete','Incomplete','Incomplete','Complete','Incomplete','Complete','Incomplete','Complete','Complete','Incomplete'],
    'Internet_Use': ['Low','High','Low','Low','Low','High','High','Low','Low','Low','High','High','Low','High'],
    'Pass': ['Yes','No','Yes','Yes','Yes','No','No','Yes','Yes','Yes','No','Yes','Yes','No']
}

df = pd.DataFrame(data)

# Entropy function
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    entropy_val = np.sum([
        (-counts[i]/np.sum(counts)) * np.log2(counts[i]/np.sum(counts))
        for i in range(len(elements))
    ])
    return entropy_val

# Information Gain
def info_gain(data, split_attribute, target="Pass"):
    total_entropy = entropy(data[target])

    vals, counts = np.unique(data[split_attribute], return_counts=True)

    weighted_entropy = np.sum([
        (counts[i]/np.sum(counts)) *
        entropy(data.where(data[split_attribute] == vals[i]).dropna()[target])
        for i in range(len(vals))
    ])

    return total_entropy - weighted_entropy

# ID3 Algorithm
def id3(data, original_data, features, target="Pass", parent_node_class=None):

    if len(np.unique(data[target])) <= 1:
        return np.unique(data[target])[0]

    elif len(data) == 0:
        return np.unique(original_data[target])[
            np.argmax(np.unique(original_data[target], return_counts=True)[1])
        ]

    elif len(features) == 0:
        return parent_node_class

    else:
        parent_node_class = np.unique(data[target])[
            np.argmax(np.unique(data[target], return_counts=True)[1])
        ]

        item_values = [info_gain(data, feature, target) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        tree = {best_feature: {}}

        features = [i for i in features if i != best_feature]

        for value in np.unique(data[best_feature]):
            sub_data = data.where(data[best_feature] == value).dropna()
            subtree = id3(sub_data, df, features, target, parent_node_class)
            tree[best_feature][value] = subtree

        return tree

# Build Decision Tree
features = df.columns[:-1]
decision_tree = id3(df, df, features)

print("Decision Tree:")
print(decision_tree)